In [0]:
%run ../Includes/Copy-Datasets

In [0]:
files = dbutils.fs.ls(f"{bookstore.dataset_path}/kafka-raw")
display(files)

In [0]:
dfRaw = spark.read.json(f"{bookstore.dataset_path}/kafka-raw")
display(dfRaw)

In [0]:
from pyspark.sql import functions as F

def process_bronze():

    schema = "key binary, value binary, topic string, partition long, offset long, timestamp long"

    query = (spark.readStream
                .format("cloudFiles")
                .option("cloudFiles.format","json")
                .schema(schema)
                .load(f"{bookstore.dataset_path}/kafka-raw")
                .withColumn("timestamp", (F.col("timestamp")/1000).cast("timestamp"))
                .withColumn("year_month", F.date_format("timestamp","yyyy-MM"))
            .writeStream
                .option("checkpointLocation", f"{bookstore.checkpoint_path}/bronze")
                .option("mergeSchema", True)
                .partitionBy("topic","year_month")
                .trigger(availableNow=True)
                .table("bronze")
            )
    
    query.awaitTermination()

In [0]:
process_bronze()

In [0]:
dfBronze = spark.table("bronze")
display(dfBronze)

In [0]:
bookstore.load_new_data()

In [0]:
process_bronze()

In [0]:
display(dfBronze.count())

In [0]:
(
    spark.readStream
        .table("bronze")
        .createOrReplaceTempView("bronze_tmp")
)

In [0]:
%sql
--order_id STRING, order_timestamp STRING, customer_id STRING, quantity BIGINT, total BIGINT,books ARRAY<STRUCT<book_id STRING, quantity BIGINT, subtotal BIGINT>>
create or replace temp view orders_silver_tmp as 
select v.* from (
select from_json(cast(value as string), 'order_id STRING, order_timestamp STRING, customer_id STRING, quantity BIGINT, total BIGINT,books ARRAY<STRUCT<book_id STRING, quantity BIGINT, subtotal BIGINT>>') as v
from bronze_tmp
where topic='orders'
);

In [0]:
query = (
    spark.table("orders_silver_tmp")
    .writeStream
    .option("checkpointLocation",f"{bookstore.checkpoint_path}/orders_silver")
    .trigger(availableNow = True)
    .table("orders_silver")
)
query.awaitTermination()

In [0]:
json_schema = 'order_id STRING, order_timestamp STRING, customer_id STRING, quantity BIGINT, total BIGINT,books ARRAY<STRUCT<book_id STRING, quantity BIGINT, subtotal BIGINT>>'

query = (
    spark.readStream.table("bronze")
    .filter("topic = 'orders'")
    .withColumn("v",F.from_json(F.col("value").cast("string"),json_schema))
    .select("v.*")
    .filter(F.col("quantity") > 0)
    .writeStream
    .option("checkpointLocation",f"{bookstore.checkpoint_path}/orders_silver")
    .trigger(availableNow = True)
    .table("orders_silver")

)
query.awaitTermination()

In [0]:
display(spark.table("orders_silver"))

In [0]:
dbutils.fs.rm(f"{bookstore.checkpoint_path}/orders_silver",True)

In [0]:
(spark.read.table("bronze").filter(F.col("topic") == 'orders').count())

In [0]:
json_schema = 'order_id STRING, order_timestamp STRING, customer_id STRING, quantity BIGINT, total BIGINT,books ARRAY<STRUCT<book_id STRING, quantity BIGINT, subtotal BIGINT>>'

batch_total = (
    spark.read.table("bronze")
    .filter("topic = 'orders'")
    .withColumn("v",F.from_json(F.col("value").cast("string"),json_schema))
    .select("v.*")
    .dropDuplicates(["order_id","order_timestamp"])
    .count()
)
print(batch_total)

In [0]:
json_schema = 'order_id STRING, order_timestamp STRING, customer_id STRING, quantity BIGINT, total BIGINT,books ARRAY<STRUCT<book_id STRING, quantity BIGINT, subtotal BIGINT>>'

dfDeduped = (
    spark.readStream.table("bronze")
    .filter("topic = 'orders'")
    .withColumn("v",F.from_json(F.col("value").cast("string"),json_schema))
    .select("v.*")
    .withColumn("order_timestamp", F.to_timestamp("order_timestamp"))
    .withWatermark("order_timestamp", "30 seconds")
    .dropDuplicates(["order_id","order_timestamp"])
)

In [0]:
def upsert_data(microBatchDF, batchId):

    microBatchDF.createOrReplaceTempView("orders_microbatch")
    sql_query = """
    merge into orders_silver t 
    using orders_microbatch s 
    on t.order_id = s.order_id and t.order_timestamp = s.order_timestamp
    when not matched then insert  *
    """
    microBatchDF.sparkSession.sql(sql_query)

In [0]:
%sql
create  or replace table  orders_silver (order_id STRING, order_timestamp timestamp, customer_id STRING, quantity BIGINT, total BIGINT,books ARRAY<STRUCT<book_id STRING, quantity BIGINT, subtotal BIGINT>>)

In [0]:
dfDeduped = (dfDeduped.withColumn("order_timestamp",F.col("order_timestamp").cast("timestamp"))
           )

In [0]:
dfDeduped.printSchema()

In [0]:
query = (
    dfDeduped.writeStream
    .foreachBatch(upsert_data)
    .option("checkpointLocation",f"{bookstore.checkpoint_path}/orders_silver")
    .trigger(availableNow=True)
    .start()
)
query.awaitTermination()

In [0]:
streaming_count = (spark.table("orders_silver").count())
print(streaming_count)
assert streaming_count == batch_total

In [0]:
%sql
merge into books_silver t
using(
  select updates.book_id as merge_key, updates.*
  from updates

  union all 

  select null as merge_key, updates.*
  from updates a  
  join books_silver b on a.book_id = b.book_id
  where b.current = true and a.price <> b.price  
) staged_updates s 
on t.book_id = s.merge_key 
when matched and t.current = true and t.price <> s.price then 
update set current = false, end_date = s.updated
when not matched then
insert (book_id, title, author, price, current, effective_date, end_date)
values (s.book_id, s.title, s.price, true, s.updated, null)

In [0]:
def type2_upsert(microBatchDF, batchId):
    microBatchDF.createOrReplaceTempView("updates")

    sql_query = """
        merge into books_silver a 
        using (
                select md5(book_id) as merge_key, updates.*
                from updates
                
                union all

                select null as merge_key, updates.*
                from updates
                join books_silver on books_silver.book_id = updates.book_id  
                where books_silver.current = true and books_silver.price <> updates.price

            ) b
            on a.merge_key = b.merge_key and a.current = true
            when matched  and a.price <> b.price 
            then update set a.current = false, a.end_date = b.updated
            when not matched then 
            insert (merge_key, book_id, title, author, price, current, effective_date, end_date)
            values (md5(b.book_id), b.book_id, b.title, b.author, b.price, true, b.updated, null)

    """
    microBatchDF.sparkSession.sql(sql_query)
    

In [0]:
%sql
CREATE or REPLACE TABLE  books_silver
(merge_key string, book_id STRING, title STRING, author STRING, price DOUBLE, current BOOLEAN, effective_date TIMESTAMP, end_date TIMESTAMP)

In [0]:
def process_books():
    schema = "book_id STRING, title STRING, author STRING, price DOUBLE, updated TIMESTAMP"

    query = (

        spark.readStream
                .table("bronze")
                .filter("topic = 'books'")
                .select(F.from_json(F.col("value").cast("string"), schema).alias("books"))
                .select("books.*")
            .writeStream
                .option("checkpointLocation", f"{bookstore.checkpoint_path}/books_silver")
                .foreachBatch(type2_upsert)
                .trigger(availableNow=True)
                .start()
    )
    query.awaitTermination()

In [0]:
bookstore.load_new_data()

In [0]:
process_books()

In [0]:
display(spark.table("books_silver"))

In [0]:
schema = "customer_id string, email string, first_name string, last_name string, gender string, street string, city string, country_code string, row_status string, row_time timestamp"

customer_df = (
    spark.table("bronze")
    .filter("topic = 'customers'")
    .withColumn("v", F.from_json(F.col("value").cast("string"),schema))
    .select("v.*")
    .filter(F.col("row_status").isin(["insert", "update"]))
)
display(customer_df)


In [0]:
from pyspark.sql.window import Window

window = Window.partitionBy("customer_id").orderBy(F.col("row_time").desc())

dfRanked = (
    customer_df.withColumn("rank", F.rank().over(window))
        .filter(F.col("rank") ==1)
        .drop("rank")
)
display(dfRanked)

In [0]:
from pyspark.sql.window import Window

def batch_upsert(microBatchDF, batchId):
    window = Window.partitionBy("customer_id").orderBy(F.col("row_time").desc())
    (microBatchDF.filter(F.col("row_status").isin(["insert","update"]))
     .withColumn("rank", F.rank().over(window))
     .filter(F.col("rank") ==1)
     .drop("rank")
     .createOrReplaceTempView("ranked_updates")
     )
    query = """
        merge into customers_silver c
        using ranked_updates r
        on c.customer_id = r.customer_id
        when matched and c.row_time < r.row_time
        then update set *
        when not matched then insert *
    """
    microBatchDF.sparkSession.sql(query)

In [0]:
%sql
create table if not exists customers_silver 
(customer_id string, email string, first_name string, last_name string, gender string, street string, city string, country string, row_time timestamp)

In [0]:
dfCountryLookup = spark.read.json(f"{bookstore.dataset_path}/country_lookup")
display(dfCountryLookup)

In [0]:
query = (
    spark.readStream
            .table("bronze")
            .filter("topic = 'customers'")
            .withColumn("v", F.from_json(F.col("value").cast("string"),schema))
            .select("v.*")
            .join(F.broadcast(dfCountryLookup), F.col("country_code") == F.col("code"), "inner")
            .writeStream
            .foreachBatch(batch_upsert)
            .option("checkpointLocation", f"{bookstore.checkpoint_path}/customers_silver")
            .trigger(availableNow=True)
            .start() 
)
query.awaitTermination()

In [0]:
display(spark.table("customers_silver"))

In [0]:
%sql
alter table customers_silver set tblproperties(delta.enableChangeDataFeed = true)

In [0]:
%sql
describe extended customers_silver

In [0]:
%sql
describe history customers_silver --2

In [0]:
%sql
select *
from table_changes("customers_silver",2)

In [0]:
dfCdf = (
    spark.readStream
            .format("delta")
            .option("readChangeData",True)
            .option("startingVersion",2)
            .table("customers_silver")
)
display(dfCdf)